# Kayfa Student Analytics — Data Cleaning & Quality Audit
### 37 Planted Issues · 4 Bonus Fixes · 3 v3 Corrections
All issues are shown **before** the fix so the problem is visible in the output, then resolved in the same cell.

## 0. Imports & Data Loading

In [1]:
import pandas as pd
import numpy as np
import json
import ast
import re
import plotly.express as px

data_path = 'student_edu_info/'

students    = pd.read_csv(data_path + 'students.csv')
groups      = pd.read_csv(data_path + 'groups.csv')
courses     = pd.read_csv(data_path + 'courses.csv')
engagement  = pd.read_csv(data_path + 'engagement_events.csv')
submissions = pd.read_csv(data_path + 'assignment_submissions.csv')
concepts    = pd.read_csv(data_path + 'concepts_performance.csv')

with open(data_path + 'grades.json', 'r') as f:
    grades_raw = json.load(f)
rows = []
for entry in grades_raw:
    for g in entry.get('grades', []):
        rows.append({
            'student_id': entry.get('student_id'),
            'course_id':  entry.get('course_id'),
            'group_id':   entry.get('group_id'),
            **g
        })
grades = pd.DataFrame(rows)

attendance_dict = pd.read_excel(data_path + 'attendance.xlsx', sheet_name=None)

print("Raw shapes loaded:")
for name, df in [('students', students), ('groups', groups), ('courses', courses),
                  ('grades', grades), ('engagement', engagement),
                  ('submissions', submissions), ('concepts', concepts)]:
    print(f"  {name:12s}: {df.shape}")
print(f"  {'attendance':12s}: {sum(len(d) for d in attendance_dict.values())} rows across {len(attendance_dict)} sheets")


Raw shapes loaded:
  students    : (502, 8)
  groups      : (12, 7)
  courses     : (7, 8)
  grades      : (5502, 10)
  engagement  : (30866, 6)
  submissions : (1504, 9)
  concepts    : (12008, 10)
  attendance  : 13010 rows across 6 sheets


## 1. students.csv — Issues 1–6

In [2]:
students.head()

,student_id,full_name,age,gender,city,email,group_id,enrollment_date
0,S0001,Hana Gamal,27,Male,Mansoura,hana.gamal@kayfa-student.io,G03,2025-12-14
1,S0002,Mona Abdelaziz,25,Female,Zagazig,mona.abdelaziz@kayfa-student.io,G06,2025-12-03
2,S0003,Menna Naguib,20,Male,Ismailia,menna.naguib@kayfa-student.io,G05,2025-12-17
3,S0004,Aya ElShafei,21,Male,Giza,aya.elshafei@kayfa-student.io,G02,2025-12-12
4,S0005,Habiba Mahmoud,24,Male,Giza,habiba.mahmoud@kayfa-student.io,G01,2025-12-14


### Issue 1 · Missing values in `full_name`

In [3]:
print("Null full_name rows:", students['full_name'].isna().sum())
print(students[students['full_name'].isna()][['student_id', 'full_name', 'group_id']])

# Issue 1: fill missing names with a placeholder
students['full_name'] = students['full_name'].fillna('Unknown Student')
print("\nAfter fix — nulls remaining:", students['full_name'].isna().sum())


Null full_name rows: 4
    student_id full_name group_id
108      S0109       NaN      G04
134      S0135       NaN      G02
286      S0287       NaN      G06
365      S0366       NaN      G03

After fix — nulls remaining: 0


### Issue 2 · Duplicate `student_id`

In [4]:
dup_ids = students[students.duplicated('student_id', keep=False)]
print(f"Duplicate student_id rows: {len(dup_ids)}")
print(dup_ids[['student_id', 'full_name', 'email']])

# Issue 2: drop duplicate student_id rows, keep first occurrence
students = students.drop_duplicates('student_id')
print(f"\nAfter fix — students: {len(students)}")


Duplicate student_id rows: 4
    student_id     full_name                          email
73       S0074   Sherif Zaki   sherif.zaki@kayfa-student.io
361      S0362  Hana Mansour  hana.mansour@kayfa-student.io
500      S0362  Hana Mansour  hana.mansour@kayfa-student.io
501      S0074   Sherif Zaki   sherif.zaki@kayfa-student.io

After fix — students: 500


### Issue 3 · Duplicate email addresses

In [5]:
dup_mask = students.duplicated('email', keep=False)
print(f"Rows sharing duplicate emails: {dup_mask.sum()}")
print(students[dup_mask][['student_id', 'full_name', 'email']].head(10).to_string(index=False))

# Issue 3: same name + same email = duplicate registration
# sort by student_id first so we always keep the earliest record deterministically
students = students.sort_values('student_id')
students = students.drop_duplicates('email', keep='first')
print(f"\nAfter fix — students: {len(students)}")


Rows sharing duplicate emails: 173
student_id       full_name                            email
     S0001      Hana Gamal      hana.gamal@kayfa-student.io
     S0005  Habiba Mahmoud  habiba.mahmoud@kayfa-student.io
     S0006     Aya ElSayed     aya.elsayed@kayfa-student.io
     S0008      Mona Gamal      mona.gamal@kayfa-student.io
     S0011      Menna Saad      menna.saad@kayfa-student.io
     S0012      Ziad Gamal      ziad.gamal@kayfa-student.io
     S0017   Tarek ElMasry   tarek.elmasry@kayfa-student.io
     S0018 Malak Abdelaziz malak.abdelaziz@kayfa-student.io
     S0022      Ziad Kamel      ziad.kamel@kayfa-student.io
     S0025    Jana ElMasry    jana.elmasry@kayfa-student.io

After fix — students: 405


### Issue 4 · Impossible age values (negative or extreme)

In [6]:
bad_age = students[(students['age'] < 15) | (students['age'] > 70)]
print(f"Impossible age rows: {len(bad_age)}")
print(bad_age[['student_id', 'age']].to_string(index=False))

fig = px.histogram(students, x='age', nbins=30, title='Age distribution (before fix)')
fig.add_vline(x=15, line_dash='dash', line_color='red', annotation_text='min=15')
fig.add_vline(x=70, line_dash='dash', line_color='red', annotation_text='max=70')
fig.show()

# Issue 4: replace impossible ages with 22 (median value)
students['age'] = np.where((students['age'] < 15) | (students['age'] > 70), 22, students['age'])
print(f"\nAfter fix — age range: {students['age'].min()} to {students['age'].max()}")


Impossible age rows: 4
student_id  age
     S0023    4
     S0209  -22
     S0453   -5
     S0478  121



After fix — age range: 17 to 31


### Issue 5 · Inconsistent gender encoding

In [7]:
print("Gender value counts before fix:")
print(students['gender'].value_counts().to_string())

# Issue 5: unify all variants to Male / Female
gender_map = {
    'MALE': 'Male', 'male': 'Male', 'M': 'Male', 'm': 'Male',
    'FEMALE': 'Female', 'female': 'Female', 'F': 'Female', 'f': 'Female', 'Fem': 'Female'
}
students['gender'] = students['gender'].replace(gender_map)
print("\nAfter fix:")
print(students['gender'].value_counts().to_string())


Gender value counts before fix:
gender
Female    214
Male      187
MALE        1
m           1
M           1
Fem         1

After fix:
gender
Female    215
Male      190


### Issue 6 · Invalid email formats

In [8]:

invalid_email_mask = ~students['email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', na=False)
print(f"Invalid email rows: {invalid_email_mask.sum()}")
print(students[invalid_email_mask][['student_id', 'email']].to_string(index=False))

# Issue 6: drop rows whose email fails the regex check
students = students[students['email'].str.contains(r'^[\w\.-]+@[\w\.-]+\.\w+$', na=False)]
print(f"\nAfter fix — students: {len(students)}")


Invalid email rows: 4
student_id            email
     S0015         missing@
     S0023        @kayfa.io
     S0061 spaces here@x.io
     S0062     not-an-email

After fix — students: 401


## 2. groups.csv — Issues 7–10 + Bonus: session_time

### Issue 7 · Duplicate `group_id`

In [9]:
dup_groups = groups[groups.duplicated('group_id', keep=False)]
print(f"Duplicate group_id rows: {len(dup_groups)}")
print(dup_groups[['group_id', 'group_name']].to_string(index=False))

# Issue 7: keep first occurrence of each group_id
groups = groups.drop_duplicates('group_id')
print(f"\nAfter fix — groups: {len(groups)}")


Duplicate group_id rows: 2
group_id      group_name
     G01 Group 01 — C001
     G01 Group 01 — C001

After fix — groups: 11


### Issue 8 · Missing instructor names

In [10]:
print(f"Null instructor rows: {groups['instructor'].isna().sum()}")
print(groups[groups['instructor'].isna()][['group_id', 'group_name', 'instructor']].to_string(index=False))

# Issue 8: fill missing instructor with a generic placeholder
groups['instructor'] = groups['instructor'].fillna('Staff')
print(f"\nAfter fix — nulls: {groups['instructor'].isna().sum()}")


Null instructor rows: 1
group_id        group_name instructor
     G99 TEST_GROUP_DELETE        NaN

After fix — nulls: 0


### Issue 9 · TEST groups removed **before** headcount recalculation

In [11]:
test_groups = groups[groups['group_name'].str.contains('TEST', na=False)]
print(f"TEST groups found: {len(test_groups)}")
print(test_groups[['group_id', 'group_name']].to_string(index=False))

# Issue 9: remove TEST groups BEFORE Issue 10 to prevent ghost groups skewing headcounts
groups = groups[~groups['group_name'].str.contains('TEST', na=False)]
print(f"\nAfter fix — groups: {len(groups)}")


TEST groups found: 1
group_id        group_name
     G99 TEST_GROUP_DELETE

After fix — groups: 10


### Issue 10 · `stated_num_students` is self-reported and wrong

In [12]:
actual_counts = students.groupby('group_id').size().reset_index(name='actual_count')
comparison = groups.merge(actual_counts, on='group_id', how='left')
comparison['actual_count'] = comparison['actual_count'].fillna(0).astype(int)
comparison['diff'] = comparison['stated_num_students'] - comparison['actual_count']
print("stated vs actual headcounts:")
print(comparison[['group_id', 'stated_num_students', 'actual_count', 'diff']].to_string(index=False))

# Issue 10: replace stated headcount with real count derived from cleaned students.csv
actual_map = students.groupby('group_id').size().to_dict()
groups['stated_num_students'] = groups['group_id'].map(actual_map).fillna(0).astype(int)
print("\nAfter fix:")
print(groups[['group_id', 'stated_num_students']].to_string(index=False))


stated vs actual headcounts:
group_id  stated_num_students  actual_count  diff
     G01                   52            35    17
     G02                   56            48     8
     G03                   67            40    27
     G04                   65            52    13
     G05                   76            37    39
     G06                   56            45    11
     G07                   70            51    19
     G08                   60            47    13
     G09                   51            43     8
     G10                   31             0    31



After fix:
group_id  stated_num_students
     G01                   35
     G02                   48
     G03                   40
     G04                   52
     G05                   37
     G06                   45
     G07                   51
     G08                   47
     G09                   43
     G10                    0


### Bonus · `session_time` inconsistent formats

In [13]:
print("session_time unique values before fix:")
print(groups['session_time'].unique())

# Bonus: normalize all session_time variants to HH:MM
import datetime

def normalize_session_time(val):
    if pd.isna(val):
        return val
    val = str(val).strip()
    if re.match(r'^\d{2}:\d{2}$', val):
        return val                                    # already HH:MM
    if re.match(r'^\d{4}$', val):
        return f'{val[:2]}:{val[2:]}'                 # 1800 -> 18:00
    for fmt in ('%I %p', '%I:%M %p'):
        try:
            return datetime.datetime.strptime(val.upper(), fmt).strftime('%H:%M')
        except ValueError:
            pass
    return val

if 'session_time' in groups.columns:
    groups['session_time'] = groups['session_time'].apply(normalize_session_time)

print("\nAfter fix:")
print(groups['session_time'].unique())


session_time unique values before fix:
['16:00' '18:00' '6 PM' '1800' '17:00' '21:00' '19:00']

After fix:
['16:00' '18:00' '17:00' '21:00' '19:00']


## 3. courses.csv — Issues 11–12

### Issue 11 · `modules` stored as stringified JSON

In [14]:
print(f"Type of modules[0] before fix: {type(courses['modules'].iloc[0]).__name__}")
print(f"Sample value: {str(courses['modules'].iloc[0])[:60]}...")

# Issue 11: parse the stringified list back into a real Python list
courses['modules'] = courses['modules'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
print(f"\nAfter fix — type: {type(courses['modules'].iloc[0]).__name__}")
print(f"Length of first modules list: {len(courses['modules'].iloc[0])}")


Type of modules[0] before fix: str
Sample value: ["Spreadsheets & Tabular Data", "Descriptive Statistics", "D...

After fix — type: list
Length of first modules list: 6


### Issue 12 · `difficulty_level` capitalization (defensive)

In [15]:
# Data confirmed clean: [Beginner, Intermediate, Advanced]
# Applied as a safeguard against future inconsistencies
print("difficulty_level values:", courses['difficulty_level'].unique())

courses['difficulty_level'] = courses['difficulty_level'].str.capitalize()
print("After fix:", courses['difficulty_level'].unique())


difficulty_level values: ['Beginner' 'Intermediate' 'Advanced']
After fix: ['Beginner' 'Intermediate' 'Advanced']


## 4. attendance.xlsx — Issues 13–19 + Bonus

### Issues 13–19 · Inconsistent status encoding across 6 sheets

In [16]:
print("Status values per sheet (raw):")
for sheet_name, df in attendance_dict.items():
    print(f"  {sheet_name}: {df['status'].unique().tolist()}")

# Issues 13-19: unify all status variants to Attended / Absent
def clean_status(val):
    val = str(val).lower().strip()
    return 'Attended' if val in ['attended', 'atttended', 'present', 'p', '1', 'yes', 'true'] else 'Absent'

all_sheets = []
for name, df in attendance_dict.items():
    if 'datetime' in df.columns:           # Issue 15: Feb sheet renamed the column
        df = df.rename(columns={'datetime': 'session_datetime'})
    df['status'] = df['status'].apply(clean_status)
    all_sheets.append(df)

attendance = pd.concat(all_sheets, ignore_index=True)
print("\nAfter fix — unique status values:", attendance['status'].unique().tolist())
print(f"Total attendance rows combined: {len(attendance)}")


Status values per sheet (raw):
  2025-12: ['attended', 'absent', 'Atttended']
  2026-01: ['Present', 'Absent']
  2026-02: [1, 0]
  2026-03: ['P', 'A']
  2026-04: ['yes', 'no']
  2026-05: [False, True]

After fix — unique status values: ['Attended', 'Absent']
Total attendance rows combined: 13010


### Bonus · Duplicate `record_id` rows in attendance

In [17]:
dup_count = attendance.duplicated('record_id').sum()
print(f"Duplicate record_ids: {dup_count}")
print(attendance[attendance.duplicated('record_id', keep=False)][['record_id', 'student_id', 'status']].head(8).to_string(index=False))

# Bonus: drop duplicate attendance records
attendance = attendance.drop_duplicates('record_id')
print(f"\nAfter fix — attendance rows: {len(attendance)}")


Duplicate record_ids: 6
record_id student_id   status
 AT002883      S0248 Attended
 AT007249      S0137 Attended
 AT007249      S0137 Attended
 AT002883      S0248 Attended
 AT010524      S0147   Absent
 AT010524      S0147   Absent
 AT005140      S0409   Absent
 AT005289      S0062   Absent

After fix — attendance rows: 13004


### Bonus · Invalid `student_id` in attendance (e.g. S9999)

In [18]:
invalid_att = attendance[~attendance['student_id'].isin(students['student_id'])]
print(f"Attendance records with invalid student_ids: {len(invalid_att)}")
print(invalid_att[['record_id', 'student_id']].head(5).to_string(index=False))

# Bonus: filter out records whose student_id does not exist in the cleaned students table
attendance_before = attendance.copy()
attendance = attendance[attendance['student_id'].isin(students['student_id'])]
print(f"\nBefore filter: {len(attendance_before)}  |  After: {len(attendance)}  |  Removed: {len(attendance_before) - len(attendance)}")


Attendance records with invalid student_ids: 2575
record_id student_id
 AT000024      S0196
 AT000026      S0239
 AT000027      S0245
 AT000028      S0262
 AT000029      S0265

Before filter: 13004  |  After: 10429  |  Removed: 2575


## 5. grades.json — Issues 20–23 + Bonus

### Issue 20 · Negative grade scores

In [19]:
neg_scores = grades[grades['score'] < 0]
print(f"Negative score rows: {len(neg_scores)}")
print(neg_scores[['student_id', 'assessment_id', 'score']].to_string(index=False))

# Issue 20: score cannot be negative — take absolute value
grades['score'] = grades['score'].abs()
print(f"\nAfter fix — min score: {grades['score'].min()}")


Negative score rows: 1
student_id assessment_id  score
     S0001       C002-QZ  -10.0

After fix — min score: 10.0


### Issue 21 · Score exceeds `max_score`

In [20]:
over_max = grades[grades['score'] > grades['max_score']]
print(f"Rows where score > max_score: {len(over_max)}")
print(over_max[['student_id', 'assessment_id', 'score', 'max_score']].to_string(index=False))

# Issue 21: cap score at max_score
grades['score'] = np.where(grades['score'] > grades['max_score'], grades['max_score'], grades['score'])
print(f"\nAfter fix — rows still over max: {(grades['score'] > grades['max_score']).sum()}")


Rows where score > max_score: 2
student_id assessment_id  score  max_score
     S0002       C004-QZ  187.0        100
     S0005       C001-QZ   78.2         10

After fix — rows still over max: 0


### Issue 22 · Missing (NaN) scores

In [21]:
null_scores = grades['score'].isna().sum()
print(f"Null score rows: {null_scores}")

# Issue 22: treat missing score as 0 — no submission means no credit
grades['score'] = grades['score'].fillna(0)
print(f"After fix — null scores: {grades['score'].isna().sum()}")


Null score rows: 2
After fix — null scores: 0


### Issue 23 · Grades referencing non-existent student IDs

In [22]:
invalid_grade_students = grades[~grades['student_id'].isin(students['student_id'])]
print(f"Grades with invalid student_id: {len(invalid_grade_students)}")

# Issue 23: referential integrity — drop grades for students not in the cleaned dataset
grades = grades[grades['student_id'].isin(students['student_id'])]
print(f"After fix — grades rows: {len(grades)}")


Grades with invalid student_id: 1089
After fix — grades rows: 4413


### Bonus · `max_score = 10` — obvious data entry error

In [23]:
suspicious = grades[grades['max_score'] < 20]
print(f"Records with max_score < 20: {len(suspicious)}")
print(suspicious[['student_id', 'assessment_id', 'score', 'max_score']].to_string(index=False))

# Bonus: correct max_score=10 to 100, then re-apply score cap
grades.loc[grades['max_score'] < 20, 'max_score'] = 100
grades['score'] = np.where(grades['score'] > grades['max_score'], grades['max_score'], grades['score'])
print(f"\nAfter fix — unique max_score values: {sorted(grades['max_score'].unique())}")


Records with max_score < 20: 1
student_id assessment_id  score  max_score
     S0005       C001-QZ   10.0         10

After fix — unique max_score values: [np.int64(100)]


## 6. engagement_events.csv — Issues 24–27

### Issue 24 · Duplicate `event_id`

In [24]:
dup_events = engagement.duplicated('event_id').sum()
print(f"Duplicate event_id rows: {dup_events}")

# Issue 24: drop duplicate event records
engagement = engagement.drop_duplicates('event_id')
print(f"After fix — engagement rows: {len(engagement)}")


Duplicate event_id rows: 8
After fix — engagement rows: 30858


### Issue 25 · Negative `duration_seconds`

In [25]:
neg_dur = engagement[engagement['duration_seconds'] < 0]
print(f"Negative duration rows: {len(neg_dur)}")
print(neg_dur[['event_id', 'event_type', 'duration_seconds']].to_string(index=False))

# Issue 25: duration cannot be negative — take absolute value
engagement['duration_seconds'] = engagement['duration_seconds'].abs()
print(f"\nAfter fix — min duration: {engagement['duration_seconds'].min()}")


Negative duration rows: 2
event_id  event_type  duration_seconds
EV000002 video_watch            -120.0
 EVBAD02 video_watch            -120.0

After fix — min duration: 20.0


### Issue 26 · Extreme `duration_seconds` outlier (> 2 hours)

In [26]:
extreme_dur = engagement[engagement['duration_seconds'] > 7200]
print(f"Duration > 7200s rows: {len(extreme_dur)}")
print(extreme_dur[['event_id', 'duration_seconds']].to_string(index=False))

# Issue 26: cap at 2 hours (7200s) — no single session should realistically exceed that
engagement['duration_seconds'] = np.where(
    engagement['duration_seconds'] > 7200, 7200, engagement['duration_seconds']
)
print(f"\nAfter fix — max duration: {engagement['duration_seconds'].max()}")


Duration > 7200s rows: 1
event_id  duration_seconds
EV000004           99999.0

After fix — max duration: 7200.0


### Issue 27 · Events before enrollment date + invalid student IDs

In [27]:
# Part A: events for students who do not exist in the cleaned dataset
invalid_eng = engagement[~engagement['student_id'].isin(students['student_id'])]
print(f"Events for non-existent student_ids: {len(invalid_eng)}")
print(invalid_eng[['event_id', 'student_id']].head(3).to_string(index=False))
engagement = engagement[engagement['student_id'].isin(students['student_id'])]

# Part B: events timestamped before the student enrolled
engagement['event_datetime'] = pd.to_datetime(engagement['event_datetime'])
students['enrollment_date']  = pd.to_datetime(students['enrollment_date'])
eng_merged = engagement.merge(students[['student_id', 'enrollment_date']], on='student_id')
pre_enroll = eng_merged[eng_merged['event_datetime'] < eng_merged['enrollment_date']]
print(f"\nEvents before enrollment date: {len(pre_enroll)}")

# Issue 27: remove both invalid-student and pre-enrollment events
engagement = eng_merged[
    eng_merged['event_datetime'] >= eng_merged['enrollment_date']
].drop(columns='enrollment_date')
print(f"After fix — engagement rows: {len(engagement)}")


Events for non-existent student_ids: 6109
event_id student_id
EV000910      S0015
EV000911      S0015
EV000912      S0015

Events before enrollment date: 1
After fix — engagement rows: 24748


## 7. assignment_submissions.csv — Issues 28–30

### Issue 28 · Wrong `is_late` flag + null `submitted_at`

In [28]:
submissions['submitted_at'] = pd.to_datetime(submissions['submitted_at'])
submissions['deadline']     = pd.to_datetime(submissions['deadline'])

# Part A: rows where submitted_at is null — is_late cannot be computed
null_sub = submissions[submissions['submitted_at'].isna()]
print(f"Null submitted_at rows: {len(null_sub)}")
print(null_sub[['submission_id', 'student_id', 'deadline', 'submitted_at', 'is_late']].to_string(index=False))

# Part B: rows where the is_late flag contradicts the actual timestamps
computed_late = submissions['submitted_at'] > submissions['deadline']
mismatch = submissions[submissions['is_late'] != computed_late]
print(f"\nis_late flag mismatches: {len(mismatch)}")
print(mismatch[['submission_id', 'submitted_at', 'deadline', 'is_late']].to_string(index=False))

# Issue 28: recompute is_late from actual timestamps; leave pd.NA where submitted_at is missing
submissions['is_late'] = submissions['submitted_at'] > submissions['deadline']
submissions['is_late'] = submissions['is_late'].astype(object)
submissions.loc[submissions['submitted_at'].isna(), 'is_late'] = pd.NA
print(f"\nAfter fix — is_late value counts (including NA):")
print(submissions['is_late'].value_counts(dropna=False).to_string())


Null submitted_at rows: 1
submission_id student_id            deadline submitted_at  is_late
     SUB00010      S0004 2026-02-20 23:59:00          NaT    False

is_late flag mismatches: 2
submission_id        submitted_at            deadline  is_late
     SUB00001 2026-02-21 18:59:00 2026-02-21 23:59:00     True
     SUB01220 2026-03-09 23:59:00 2026-03-09 23:59:00     True

After fix — is_late value counts (including NA):
is_late
False    965
True     538
<NA>       1


### Issue 29 · Impossible `attempts` (0 or negative)

In [29]:
bad_attempts = submissions[submissions['attempts'] < 1]
print(f"Attempts < 1 rows: {len(bad_attempts)}")
print(f"Min attempts in data: {submissions['attempts'].min()}")

# Issue 29 (defensive): you cannot submit 0 or fewer times — clip to minimum of 1
submissions['attempts'] = submissions['attempts'].clip(lower=1)
print(f"After fix — min attempts: {submissions['attempts'].min()}")


Attempts < 1 rows: 0
Min attempts in data: 1
After fix — min attempts: 1


### Issue 30 · Negative `time_spent_minutes`

In [30]:
neg_time = submissions[submissions['time_spent_minutes'] < 0]
print(f"Negative time_spent rows: {len(neg_time)}")
print(neg_time[['submission_id', 'time_spent_minutes']].to_string(index=False))

# Issue 30: time cannot be negative — take absolute value
submissions['time_spent_minutes'] = submissions['time_spent_minutes'].abs()
print(f"\nAfter fix — min time_spent: {submissions['time_spent_minutes'].min()}")


Negative time_spent rows: 1
submission_id  time_spent_minutes
     SUB00006                 -40

After fix — min time_spent: 10


## 8. concepts_performance.csv — Issues 31–33

### Issue 31 · `score_pct` outside [0, 100]

In [31]:
out_of_range = concepts[(concepts['score_pct'] < 0) | (concepts['score_pct'] > 100)]
print(f"Out-of-range score_pct rows: {len(out_of_range)}")
print(out_of_range[['record_id', 'student_id', 'score_pct', 'mastery_status']].to_string(index=False))

# Issue 31: clip score_pct to valid range [0, 100]
concepts['score_pct'] = concepts['score_pct'].clip(0, 100)
print(f"\nAfter fix — score_pct range: {concepts['score_pct'].min()} to {concepts['score_pct'].max()}")


Out-of-range score_pct rows: 2
record_id student_id  score_pct mastery_status
  CPBAD02      S0009      -33.0         failed
  CPBAD03      S0013      142.0         failed

After fix — score_pct range: 0.0 to 100.0


### Issue 32 · `mastery_status` contradicts `score_pct` (threshold = 60, not 50)

In [32]:
# Correct threshold confirmed from data is 60.
# Using 50 would wrongly promote ~1,761 students (score 50-60) to 'passed'
MASTERY_THRESHOLD = 60
expected = np.where(concepts['score_pct'] >= MASTERY_THRESHOLD, 'passed', 'failed')
contradictions = concepts[concepts['mastery_status'] != expected]
print(f"Mastery contradictions at threshold={MASTERY_THRESHOLD}: {len(contradictions)}")
print(contradictions[['record_id', 'score_pct', 'mastery_status']].head(5).to_string(index=False))

# Issue 32: recompute mastery_status using the correct threshold of 60
concepts['mastery_status'] = np.where(
    concepts['score_pct'] >= MASTERY_THRESHOLD, 'passed', 'failed'
)
print(f"\nAfter fix — mastery_status counts:")
print(concepts['mastery_status'].value_counts().to_string())


Mastery contradictions at threshold=60: 14
record_id  score_pct mastery_status
 CP000768       60.0         failed
 CP001132       60.0         failed
 CP003730       60.0         failed
 CP005521       60.0         failed
 CP006376       60.0         failed

After fix — mastery_status counts:
mastery_status
passed    9148
failed    2860


### Issue 33 · Duplicate `record_id` in concepts

In [33]:
dup_concepts = concepts.duplicated('record_id').sum()
print(f"Duplicate record_id rows: {dup_concepts}")

# Issue 33: drop duplicate concept records
concepts = concepts.drop_duplicates('record_id')
print(f"After fix — concepts rows: {len(concepts)}")


Duplicate record_id rows: 5
After fix — concepts rows: 12003


## 9. Cross-file Logical Issues — Issues 34–37

### Issue 34 · `group_id` in grades does not match student's actual group

In [34]:
student_group_map = students.set_index('student_id')['group_id'].to_dict()
grades['student_group'] = grades['student_id'].map(student_group_map)
mismatch_mask = grades['group_id'] != grades['student_group']
real_mismatches = grades[mismatch_mask & grades['student_group'].notna()]
print(f"Grade records with wrong group_id: {len(real_mismatches)}")
if len(real_mismatches):
    print(real_mismatches[['student_id', 'group_id', 'student_group']].head(5).to_string(index=False))

# Issue 34: drop grades where group_id does not match the student's enrolled group
grades = grades[~mismatch_mask].drop(columns='student_group')
print(f"\nAfter fix — grades rows: {len(grades)}")


Grade records with wrong group_id: 33
student_id group_id student_group
     S0014      G01           GZZ
     S0014      G01           GZZ
     S0014      G01           GZZ
     S0014      G01           GZZ
     S0014      G01           GZZ

After fix — grades rows: 4380


### Bonus · Duplicate (student, assessment) pairs in grades

In [35]:
dup_pairs = grades.duplicated(subset=['student_id', 'assessment_id']).sum()
print(f"Duplicate (student_id, assessment_id) pairs: {dup_pairs}")

# Bonus: keep the best (highest) score per student per assessment
before = len(grades)
grades = grades.sort_values('score', ascending=False).drop_duplicates(
    subset=['student_id', 'assessment_id'], keep='first'
)
print(f"Removed: {before - len(grades)} rows  |  Grades remaining: {len(grades)}")


Duplicate (student_id, assessment_id) pairs: 2388
Removed: 2388 rows  |  Grades remaining: 1992


### Issue 35 · Female names tagged as Male

In [36]:
female_names = ['Hana', 'Menna', 'Aya', 'Sara', 'Fatma', 'Nour', 'Mariam', 'Habiba']
pattern = '|'.join(female_names)
wrong_gender = (
    students['full_name'].str.contains(pattern, case=False, na=False) &
    (students['gender'] == 'Male')
)
print(f"Female names tagged as Male: {wrong_gender.sum()}")
print(students[wrong_gender][['student_id', 'full_name', 'gender']].head(8).to_string(index=False))

# Issue 35: correct gender for known female names
students.loc[wrong_gender, 'gender'] = 'Female'
print(f"\nAfter fix — gender counts:")
print(students['gender'].value_counts().to_string())


Female names tagged as Male: 32
student_id      full_name gender
     S0001     Hana Gamal   Male
     S0003   Menna Naguib   Male
     S0004   Aya ElShafei   Male
     S0005 Habiba Mahmoud   Male
     S0032      Hana Awad   Male
     S0058   Sara Darwish   Male
     S0060      Sara Awad   Male
     S0075   Hana Mahmoud   Male

After fix — gender counts:
gender
Female    244
Male      157


### Issue 36 · Students with non-existent `group_id`

In [37]:
invalid_group_mask = ~students['group_id'].isin(groups['group_id'])
print(f"Students with invalid group_id: {invalid_group_mask.sum()}")
print(students[invalid_group_mask][['student_id', 'full_name', 'group_id']].to_string(index=False))

# Issue 36: referential integrity — remove students pointing to a non-existent group
students = students[~invalid_group_mask]
print(f"\nAfter fix — students: {len(students)}")


Students with invalid group_id: 3
student_id    full_name group_id
     S0014   Ali Refaat      GZZ
     S0095 Yasmin Fathy      G77
     S0323    Aya Kamel      G77

After fix — students: 398


### Issue 37 · Concepts records for students not in cleaned dataset

In [38]:
orphan_concepts = ~concepts['student_id'].isin(students['student_id'])
print(f"Concepts rows for unknown students: {orphan_concepts.sum()}")

# Issue 37: referential integrity — drop orphan concept records
concepts = concepts[~orphan_concepts]
print(f"After fix — concepts rows: {len(concepts)}")


Concepts rows for unknown students: 2448
After fix — concepts rows: 9555


## 10. Final Summary

In [39]:
print("=" * 45)
print("CLEAN DATASET SHAPES")
print("=" * 45)
for name, df in [
    ('students',    students),
    ('groups',      groups),
    ('courses',     courses),
    ('attendance',  attendance),
    ('grades',      grades),
    ('engagement',  engagement),
    ('submissions', submissions),
    ('concepts',    concepts),
]:
    print(f"  {name:12s}: {str(df.shape):>15}")
print("=" * 45)


CLEAN DATASET SHAPES
  students    :        (398, 8)
  groups      :         (10, 7)
  courses     :          (7, 8)
  attendance  :      (10429, 6)
  grades      :      (1992, 10)
  engagement  :      (24748, 6)
  submissions :       (1504, 9)
  concepts    :      (9555, 10)


In [40]:
## Step 3 — Join Across Sources

# 1. Student base: students ← groups ← courses
student_full = students.merge(
    groups[['group_id', 'group_name', 'course_id', 'instructor', 'session_day']],
    on='group_id', how='left'
).merge(
    courses[['course_id', 'course_name', 'category', 'difficulty_level']],
    on='course_id', how='left'
)

# 2. Avg grade per student
grades['pct'] = grades['score'] / grades['max_score'] * 100
avg_grade = grades.groupby('student_id')['pct'].mean().reset_index(name='avg_grade_pct')

# 3. Attendance rate per student
att_rate = attendance.groupby('student_id').apply(
    lambda x: (x['status']=='Attended').sum() / len(x) * 100,
    include_groups=False
).reset_index(name='attendance_rate')
# 4. Engagement per student
eng_agg = engagement.groupby('student_id').agg(
    login_count=('event_type', lambda x: (x == 'login').sum()),
    video_watch_seconds=('duration_seconds', 'sum')
).reset_index()
eng_agg['video_watch_minutes'] = eng_agg['video_watch_seconds'] / 60

# 5. Failed concepts count per student
failed_concepts = (
    concepts[concepts['mastery_status'] == 'failed']
    .groupby('student_id').size()
    .reset_index(name='failed_concepts')
)

# 6. Master join
master = (
    student_full
    .merge(avg_grade,      on='student_id', how='left')
    .merge(att_rate,       on='student_id', how='left')
    .merge(eng_agg,        on='student_id', how='left')
    .merge(failed_concepts, on='student_id', how='left')
)

master['failed_concepts']    = master['failed_concepts'].fillna(0)
master['avg_grade_pct']      = master['avg_grade_pct'].fillna(0)
master['attendance_rate']    = master['attendance_rate'].fillna(0)
master['login_count']        = master['login_count'].fillna(0)
master['video_watch_minutes'] = master['video_watch_minutes'].fillna(0)

print(f"Master table shape: {master.shape}")
print(master[['student_id', 'full_name', 'course_name',
              'avg_grade_pct', 'attendance_rate',
              'login_count', 'failed_concepts']].head())

Master table shape: (398, 21)
  student_id       full_name                  course_name  avg_grade_pct  \
0      S0001      Hana Gamal           Python Programming          82.34   
1      S0002  Mona Abdelaziz                 UI/UX Design          88.82   
2      S0003    Menna Naguib              Web Development          77.34   
3      S0004    Aya ElShafei  Data Analytics Fundamentals          83.80   
4      S0005  Habiba Mahmoud  Data Analytics Fundamentals          71.68   

   attendance_rate  login_count  failed_concepts  
0        80.769231           16              6.0  
1        92.307692           32              0.0  
2        84.615385           27              2.0  
3        84.615385           27              0.0  
4        73.076923           17              7.0  


In [41]:
master.to_csv('master_data_final.csv', index=False)


In [42]:
master_data_final = pd.read_csv('master_data_final.csv')
master_data_final.head()

,student_id,full_name,age,gender,city,email,group_id,enrollment_date,group_name,course_id,...,session_day,course_name,category,difficulty_level,avg_grade_pct,attendance_rate,login_count,video_watch_seconds,video_watch_minutes,failed_concepts
0,S0001,Hana Gamal,27,Female,Mansoura,hana.gamal@kayfa-student.io,G03,2025-12-14,Group 03 — C002,C002,...,Sunday,Python Programming,Programming,Beginner,82.34,80.769231,16,17787.0,296.450000,6.0
1,S0002,Mona Abdelaziz,25,Female,Zagazig,mona.abdelaziz@kayfa-student.io,G06,2025-12-03,Group 06 — C004,C004,...,Saturday,UI/UX Design,Design,Beginner,88.82,92.307692,32,18570.0,309.500000,0.0
2,S0003,Menna Naguib,20,Female,Ismailia,menna.naguib@kayfa-student.io,G05,2025-12-17,Group 05 — C003,C003,...,Tuesday,Web Development,Programming,Intermediate,77.34,84.615385,27,11317.0,188.616667,2.0
3,S0004,Aya ElShafei,21,Female,Giza,aya.elshafei@kayfa-student.io,G02,2025-12-12,Group 02 — C001,C001,...,Thursday,Data Analytics Fundamentals,Analytics,Beginner,83.80,84.615385,27,17301.0,288.350000,0.0
4,S0005,Habiba Mahmoud,24,Female,Giza,habiba.mahmoud@kayfa-student.io,G01,2025-12-14,Group 01 — C001,C001,...,Thursday,Data Analytics Fundamentals,Analytics,Beginner,71.68,73.076923,17,11072.0,184.533333,7.0
